In [1]:
import json

In [2]:
META_SETTINGS_FILE_PATH = "./meta_settings.jsonl"
EXP_RES_FILE_PATH = "./exp_res.jsonl"

In [3]:
meta_setting_template = {
    "label": str(1).zfill(3),
    "rag_basic_settings": {
        "retrieval": {
            "method": "mmr",
            "top_k": 10,
            "fetch_k": 40,
            "score_threshold": 0.75,
            "top_n": 5,
            "embed": {
                "provider": "hf",
                "model_name": "bge-large-en-v1.5",
                "retrival_database_batch_size": 256
            }
        },
        "reranker": {
            "model": "bge-reranker-large",
        },
        "extractor": {
            "model": "bge-large-en-v1.5",
        }
    },
    "exp_settings": {
        "attack":{
            "query": "ikea",
            "template": "tgtb",
        },
        "generator": "o4-mini",
        "rag":{
            "optional_setting": {
                "reranker": True,
                "rewriter": False,
                "extractor": False
            }
        },
        "tool_llm": "4.1-mini"
    }
}

In [11]:
exp_res_template = {
    "meta_label": str(1).zfill(3),
    "dataset": "scifact",
    "attack_num": 200,
    "retrieval_unique_num": None,
    "leakage_unique_num":{
        "recall_ge_03": None,
        "recall_ge_05": None,
        "recall_ge_07": None,
        "recall_ge_09": None
    },
    "attack_success_num":{
        "recall_ge_03": None,
        "recall_ge_05": None,
        "recall_ge_07": None,
        "recall_ge_09": None
    },
    "repeat_quality":{
        "recall_ge_05": None
    },
    "avg_semantic_similarity": {
        "len_weighted_ss_05": None,
        "len_weighted_ss_07": None,
        "len_weighted_ss_09": None
    }
}

In [5]:
def load_jsonl_file(file_path):
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return data

META_list = load_jsonl_file(META_SETTINGS_FILE_PATH)
EXP_RES_list = load_jsonl_file(EXP_RES_FILE_PATH)

def set_meta_info_no_label(attack, generator, if_reranker = False, if_rewriter = False, if_extractor = False, template = None):
    meta_info = meta_setting_template.copy()
    meta_info.pop("label")
    meta_info["exp_settings"]["attack"]["query"] = attack
    meta_info["exp_settings"]["attack"]["template"] = template if template else attack
    meta_info["exp_settings"]["generator"] = generator
    meta_info["exp_settings"]["rag"]["optional_setting"]["reranker"] = if_reranker
    meta_info["exp_settings"]["rag"]["optional_setting"]["rewriter"] = if_rewriter
    meta_info["exp_settings"]["rag"]["optional_setting"]["extractor"] = if_extractor
    return meta_info

def find_meta_info_label(meta_info):
    for item in META_list:
        if item["exp_settings"] == meta_info["exp_settings"] and \
            item["rag_basic_settings"] == meta_info["rag_basic_settings"]:
            return item["label"]
    label = len(META_list) + 1
    META_list.append({
        "label": str(label).zfill(3),
        **meta_info
    })
    print(f"[INFO] New meta info added with label: {str(label).zfill(3)}")
    save_meta_list()
    return str(label).zfill(3)

def save_meta_list():
    with open(META_SETTINGS_FILE_PATH, 'w') as f:
        for item in META_list:
            f.write(json.dumps(item) + '\n')

def save_exp_res_list():
    with open(EXP_RES_FILE_PATH, 'w') as f:
        for item in EXP_RES_list:
            f.write(json.dumps(item) + '\n')

def create_update_exp_res(meta_label, dataset, metric, metric_sub, value):
    for item in EXP_RES_list:
        if item["meta_label"] == meta_label and item["dataset"] == dataset:
            if metric_sub:
                item[metric][metric_sub] = value
            else:
                item[metric] = value
            # print(f"[INFO] Exp result updated for meta label: {meta_label}, dataset: {dataset}")
            save_exp_res_list()
            return item
    new_item = exp_res_template.copy()
    new_item["meta_label"] = meta_label
    new_item["dataset"] = dataset
    if metric_sub:
        new_item[metric][metric_sub] = value
    else:
        new_item[metric] = value
    EXP_RES_list.append(new_item)
    print(f"[INFO] New exp result added for meta label: {meta_label}, dataset: {dataset}")
    save_exp_res_list()
    return new_item

def get_exp_res(meta_label, dataset):
    for item in EXP_RES_list:
        if item["meta_label"] == meta_label and item["dataset"] == dataset:
            return item
    return None

def find_exp_res(attack, generator, if_reranker, if_rewriter, if_extractor, template, dataset):
    meta_info = set_meta_info_no_label(attack, generator, if_reranker, if_rewriter, if_extractor, template)
    meta_label = find_meta_info_label(meta_info)
    exp_res = get_exp_res(meta_label, dataset)
    if exp_res:
        return exp_res
    return None

def find_potential_meta_info_label(attack, generator, template):
    potential_labels = []
    for item in META_list:
        if item["exp_settings"]["attack"]["query"] == attack and \
            item["exp_settings"]["attack"]["template"] == template and \
            item["exp_settings"]["generator"] == generator:
            potential_labels.append(item["label"])
    return potential_labels

In [6]:
def update_all(attack, generator, if_reranker, if_rewriter, if_extractor, template, dataset, metric, metric_sub, value):
    meta_info = set_meta_info_no_label(
        attack=attack,
        generator=generator,
        if_reranker=if_reranker,
        if_rewriter=if_rewriter,
        if_extractor=if_extractor,
        template=template
    )
    m_label = find_meta_info_label(meta_info)
    item= create_update_exp_res(
        meta_label=m_label,
        dataset=dataset,
        metric=metric,
        metric_sub=metric_sub,
        value=value
    )
    return item

In [7]:
def save_from_list(list, attack="IKEA", generator="o4-mini", if_reranker=True, if_rewriter=False, if_extractor=False, template="TGTB", dataset="scifact"):
    meta_info = set_meta_info_no_label(
        attack=attack,
        generator=generator,
        if_reranker=if_reranker,
        if_rewriter=if_rewriter,
        if_extractor=if_extractor,
        template=template
    )
    m_label = find_meta_info_label(meta_info)
    create_update_exp_res(
        meta_label=m_label,
        dataset=dataset,
        metric="retrieval_unique_num",
        metric_sub=None,
        value=list[0]
    )
    create_update_exp_res(
        meta_label=m_label,
        dataset=dataset,
        metric="leakage_unique_num",
        metric_sub="recall_ge_05",
        value=list[1]
    )
    create_update_exp_res(
        meta_label=m_label,
        dataset=dataset,
        metric="attack_success_num",
        metric_sub="recall_ge_05",
        value=list[2]
    )
    create_update_exp_res(
        meta_label=m_label,
        dataset=dataset,
        metric="repeat_quality",
        metric_sub="recall_ge_05",
        value=list[3]
    )
    create_update_exp_res(
        meta_label=m_label,
        dataset=dataset,
        metric="avg_semantic_similarity",
        metric_sub=None,
        value=list[4]
    )
    create_update_exp_res(
        meta_label=m_label,
        dataset=dataset,
        metric="leakage_unique_num",
        metric_sub="recall_ge_03",
        value=list[5]
    )
    create_update_exp_res(
        meta_label=m_label,
        dataset=dataset,
        metric="attack_success_num",
        metric_sub="recall_ge_03",
        value=list[6]
    )
    create_update_exp_res(
        meta_label=m_label,
        dataset=dataset,
        metric="leakage_unique_num",
        metric_sub="recall_ge_07",
        value=list[7]
    )
    create_update_exp_res(
        meta_label=m_label,
        dataset=dataset,
        metric="attack_success_num",
        metric_sub="recall_ge_07",
        value=list[8]
    )
    create_update_exp_res(
        meta_label=m_label,
        dataset=dataset,
        metric="leakage_unique_num",
        metric_sub="recall_ge_09",
        value=list[9]
    )
    create_update_exp_res(
        meta_label=m_label,
        dataset=dataset,
        metric="attack_success_num",
        metric_sub="recall_ge_09",
        value=list[10]
    )
    

In [295]:
setting_now = ['por', None, 'qwen2_5-14b-instruct', True, True, False, 'nfcorpus']
list = [529, 467, 191, 0.9783006769043537, 0.7914754483103752, 471, 191, 463, 186, 463, 186]

In [296]:
exp_res_now = find_exp_res(
    attack=setting_now[0],
    template=setting_now[1],
    generator=setting_now[2],
    if_rewriter=setting_now[3],
    if_reranker=setting_now[4],
    if_extractor=setting_now[5],
    dataset=setting_now[6]
)
if exp_res_now:
    print(f"[INFO] Existing experiment result found: {exp_res_now}")
else:
    print(f"[INFO] No existing experiment result found. Creating new entry.")
    # Add code here to create a new experiment result entry if needed
    META_list = load_jsonl_file(META_SETTINGS_FILE_PATH)
    EXP_RES_list = load_jsonl_file(EXP_RES_FILE_PATH)
    save_from_list(
        list,
        attack=setting_now[0],
        template=setting_now[1],
        generator=setting_now[2],
        if_rewriter=setting_now[3],
        if_reranker=setting_now[4],
        if_extractor=setting_now[5],
        dataset=setting_now[6]
    )

[INFO] No existing experiment result found. Creating new entry.
[INFO] New exp result added for meta label: 018, dataset: nfcorpus


In [9]:
label_list = find_potential_meta_info_label("por", "qwen2_5-14b-instruct", "por")
dataset = "nfcorpus"

In [10]:
exp_needed = []
for i in EXP_RES_list:
    if i["meta_label"] in label_list and i["dataset"] == dataset:
        exp_needed.append(i)

print(f"[INFO] Found {len(exp_needed)} experiment results matching criteria.")
if exp_needed:
    print(
        "ASR: ",sum([i["attack_success_num"]["recall_ge_05"] for i in exp_needed])/len(exp_needed)/200,"\n",
    )
    print(
        "UCL: ",sum([i["leakage_unique_num"]["recall_ge_05"] for i in exp_needed])/len(exp_needed)/1000,"\n",
    )
    print(
        "ARE: ",sum([i["retrieval_unique_num"] for i in exp_needed])/len(exp_needed)/1000,"\n",
    )
    print(
        "COR: ", sum([i["repeat_quality"]["recall_ge_05"] for i in exp_needed])/len(exp_needed),"\n",
    )

[INFO] Found 3 experiment results matching criteria.
ASR:  0.9716666666666667 

UCL:  0.5273333333333333 

ARE:  0.575 

COR:  0.9839718102688467 

